# 99 - Robustness Checks

Sensitivity of IV estimates to: bandwidth, instrument definition, sample restrictions, and alternative outcomes.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../data/processed")
OUT_DIR  = Path("../outputs")
OUT_DIR.mkdir(exist_ok=True)

PANEL_FILE = "analysis_panel.parquet"
if not (DATA_DIR / PANEL_FILE).exists():
    raise FileNotFoundError(
        "Analysis panel not found. Build it by running in order:\n"
        "  python scripts/download_epa_aqs.py --email EMAIL --key KEY\n"
        "  python scripts/download_hms_smoke.py\n"
        "  python scripts/download_seda.py  (manual — see instructions)\n"
        "  python src/merge/build_crosswalks.py\n"
        "  python src/ingest/epa_aqs.py\n"
        "  python src/ingest/seda.py\n"
        "  python src/exposure/smoke_instrument.py\n"
        "  python src/merge/build_panel.py"
    )

panel = pd.read_parquet(DATA_DIR / PANEL_FILE)
print(f"Panel: {panel.shape}")
print(f"Districts: {panel['leaid'].nunique()}")
print(f"Years: {sorted(panel['year'].dropna().unique())}")

In [ ]:
from linearmodels.iv import IV2SLS
from linearmodels.panel import PanelOLS

In [ ]:
panel_iv = panel.dropna(subset=["leaid","year","pm25_annual_mean","smoke_days","test_score_mean"]).copy()
panel_iv["year"] = panel_iv["year"].astype(int)

def demean(df, cols):
    for col in cols:
        df[f"{col}_dm"] = (
            df[col]
            - df.groupby("leaid")[col].transform("mean")
            - df.groupby("year")[col].transform("mean")
            + df[col].mean()
        )
    return df

panel_iv = demean(panel_iv, ["test_score_mean","pm25_annual_mean","smoke_days"])

def run_iv(data):
    try:
        res = IV2SLS(
            dependent=data["test_score_mean_dm"],
            exog=None,
            endog=data[["pm25_annual_mean_dm"]],
            instruments=data[["smoke_days_dm"]],
        ).fit(cov_type="robust")
        b  = res.params["pm25_annual_mean_dm"]
        ci = res.conf_int().loc["pm25_annual_mean_dm"]
        return b, ci["lower"], ci["upper"]
    except Exception as e:
        return np.nan, np.nan, np.nan

b_base, lo, hi = run_iv(panel_iv)
print(f"Baseline IV: {b_base:.4f}  [{lo:.4f}, {hi:.4f}]")

## Check 1 - Drop 2018 (Camp Fire year)

In [ ]:
sub = demean(panel_iv[panel_iv["year"]!=2018].copy(), ["test_score_mean","pm25_annual_mean","smoke_days"])
b, lo_, hi_ = run_iv(sub)
print(f"Drop 2018: {b:.4f}  [{lo_:.4f}, {hi_:.4f}]")
print(f"Baseline:  {b_base:.4f}")

## Check 2 - Drop California

In [ ]:
if "state" in panel_iv.columns:
    sub = demean(panel_iv[panel_iv["state"]!="CA"].copy(), ["test_score_mean","pm25_annual_mean","smoke_days"])
    b, lo_, hi_ = run_iv(sub)
    print(f"Drop CA: {b:.4f}  [{lo_:.4f}, {hi_:.4f}]")
else:
    print("State column not in panel — add from SEDA geodata for this check")

## Check 3 - Heavy smoke instrument only

In [ ]:
if "smoke_days_heavy" in panel_iv.columns:
    sub = demean(panel_iv.copy(), ["test_score_mean","pm25_annual_mean","smoke_days_heavy"])
    sub = sub.rename(columns={"smoke_days_heavy_dm":"smoke_days_dm"})
    b, lo_, hi_ = run_iv(sub)
    print(f"Heavy smoke only: {b:.4f}  [{lo_:.4f}, {hi_:.4f}]")
else:
    print("smoke_days_heavy not available — check smoke instrument build step")

## Summary

In [ ]:
checks = [
    ("Baseline IV (all smoke, all years)", b_base, lo, hi),
]
print(f"{'Specification':<45}  {'β':>8}  CI")
print("-"*70)
for name, b, lo_, hi_ in checks:
    if not np.isnan(b):
        print(f"{name:<45}  {b:>8.4f}  [{lo_:+.4f}, {hi_:+.4f}]")